# Bob — Polaris STS AssumeRole demo

Bob's catalog role has table-scoped grants on `demo.order_items` (shared with
Alice), `demo.products`, and `demo.events`. He has **no** grant on
`demo.customers` or `demo.orders` — the last cell proves that matters.

In [ ]:
from dotenv import dotenv_values

creds = dotenv_values('/home/jovyan/creds/bob.env')
CLIENT_ID = creds['POLARIS_CLIENT_ID']
CLIENT_SECRET = creds['POLARIS_CLIENT_SECRET']
CATALOG_NAME = creds['POLARIS_CATALOG_NAME']
assert CLIENT_ID != 'CHANGE_ME', 'polaris-setup has not run yet — see ~/bw/sts/runbook.md step 6/7'

# Separate, narrowly-scoped VASTDB key — NOT vended by Polaris (Generic
# Tables are metadata-only in Polaris 1.7.0). See runbook.md's VASTDB
# extension section for why these are two unrelated credential paths.
db_creds = dotenv_values('/home/jovyan/creds/bob_db.env')
DB_ACCESS_KEY = db_creds['VASTDB_ACCESS_KEY_ID']
DB_SECRET_KEY = db_creds['VASTDB_SECRET_ACCESS_KEY']
assert DB_ACCESS_KEY != 'CHANGE_ME', 'bob-db VAST key not filled in yet — see runbook.md VASTDB extension, VMS step'

# Shared VAST DB endpoint(s) — see runbook.md Prerequisites.
endpoint_creds = dotenv_values('/home/jovyan/creds/vastdb_endpoint.env')
VASTDB_ENDPOINT = endpoint_creds['VASTDB_ENDPOINT']
VASTDB_DATA_ENDPOINTS = endpoint_creds['VASTDB_DATA_ENDPOINTS']

print(f'Catalog: {CATALOG_NAME}, principal: bob ({CLIENT_ID})')

In [ ]:
from pyspark.sql import SparkSession

# NOTE: spark.driver.extraClassPath / spark.executor.extraClassPath for the
# VAST connector jars are set in the image's static spark-defaults.conf,
# NOT here — that config cannot take effect from SparkSession.builder in
# client/local mode (the driver JVM has already started by this point).

spark = (
    SparkSession.builder.appName('bob-sts-demo')
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', 'http://polaris:8181/api/catalog')
    .config('spark.sql.catalog.polaris.warehouse', CATALOG_NAME)
    .config('spark.sql.catalog.polaris.credential', f'{CLIENT_ID}:{CLIENT_SECRET}')
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.header.X-Iceberg-Access-Delegation', 'vended-credentials')
    .config('spark.sql.defaultCatalog', 'polaris')
    # VAST DB ("ndb") catalog — a second, unrelated catalog in the same
    # session. Auth here is bob-db's own static key, not vended.
    .config('spark.sql.catalog.ndb', 'spark.sql.catalog.ndb.VastCatalog')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,ndb.NDBSparkSessionExtension')
    .config('spark.ndb.endpoint', VASTDB_ENDPOINT)
    .config('spark.ndb.data_endpoints', VASTDB_DATA_ENDPOINTS)
    .config('spark.ndb.access_key_id', DB_ACCESS_KEY)
    .config('spark.ndb.secret_access_key', DB_SECRET_KEY)
    # Spark's local-mode default (4) trips the VAST connector's write-safety
    # guard ("Unsafe Spark configuration blocks write") and, worse, lets a
    # failed task silently retry a non-idempotent INSERT — confirmed live via
    # duplicate rows. Must be 1 for both the guard and correctness.
    .config('spark.task.maxFailures', '1')
    .getOrCreate()
)
spark.sql('SHOW NAMESPACES').show()

In [ ]:
spark.sql("SHOW TABLES IN ndb.`whitmore`.`polaris_demo_db`").show(truncate=False)

# SHOW TABLE EXTENDED is not supported for v2 tables, since the
# Polaris/Iceberg REST catalog is a v2 catalog and that legacy command
# only supports v1/Hive-style catalogs.
NAMESPACE = 'demo'

rows = []
for t in spark.sql(f"SHOW TABLES IN {NAMESPACE}").collect():
    try:
        desc = spark.sql(f"DESCRIBE TABLE EXTENDED {NAMESPACE}.{t.tableName}").collect()
        location = next(r.data_type for r in desc if r.col_name == 'Location')
    except Exception:
        # Alice/Bob don't have grants on every table in this namespace —
        # DESCRIBE also needs LOAD_TABLE_WITH_READ_DELEGATION (see runbook
        # bug #4), so it 403s here exactly like a real read/write would.
        location = '<no access>'
    rows.append((NAMESPACE, t.tableName, location))

spark.createDataFrame(rows, ['namespace', 'table', 'location']).show(truncate=False)

In [ ]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS demo.products (
    product_id INT, name STRING, price DOUBLE
  ) USING iceberg
''')
spark.sql('''
  CREATE TABLE IF NOT EXISTS demo.events (
    event_id INT, description STRING
  ) USING iceberg
''')
spark.sql('''
  CREATE TABLE IF NOT EXISTS demo.order_items (
    order_id INT, item STRING, qty INT, written_by STRING
  ) USING iceberg
''')

spark.sql("INSERT INTO demo.products VALUES (1, 'Widget', 9.99), (2, 'Gadget', 19.99)")
spark.sql("INSERT INTO demo.events VALUES (1, 'restock')")
spark.sql("INSERT INTO demo.order_items VALUES (101, 'gadget', 1, 'bob')")

print('Bob-owned tables:')
spark.sql('SELECT * FROM demo.products').show()
spark.sql('SELECT * FROM demo.events').show()

In [ ]:
# demo.order_items is the deliberate overlap with Alice — this should show
# Alice's earlier row (written_by='alice') alongside Bob's own.
spark.sql('SELECT * FROM demo.order_items ORDER BY order_id').show()

In [ ]:
# Pass/fail auth check: Bob has no grant on demo.customers, so this write
# must be REJECTED by Polaris.
try:
    spark.sql('CREATE TABLE IF NOT EXISTS demo.customers (customer_id INT, name STRING) USING iceberg')
    spark.sql("INSERT INTO demo.customers VALUES (999, 'should not be allowed')")
    print('FAIL: Bob was able to write to demo.customers — isolation is broken')
except Exception as e:
    print('PASS: write to demo.customers correctly rejected')
    print(type(e).__name__, str(e)[:300])

## VASTDB extension

The `whitmore` bucket is also a VAST Database. These tables (`shipments`,
`reviews`, `inventory_events`) are registered in Polaris as **Generic
Tables** for unified catalog discovery only — Polaris 1.7.0 does not vend
credentials for them. Actual read/write access here is via the VAST DB
Spark connector (`ndb` catalog), authenticated with bob-db's own separate,
narrowly-scoped VAST key — isolation is enforced by VAST's Identity Policy
on that key, not by Polaris.

In [ ]:
DB = "ndb.`whitmore`.`polaris_demo_db`"

spark.sql(f"SHOW TABLES IN {DB}").show()

spark.sql(f"INSERT INTO {DB}.reviews VALUES (1, 1, 5, 'Great widget'), (2, 2, 4, 'Solid gadget')")
spark.sql(f"INSERT INTO {DB}.inventory_events VALUES (2, 'gadget', 1, 'bob')")

print('Bob-owned VASTDB tables:')
spark.sql(f"SELECT * FROM {DB}.reviews").show()

In [ ]:
# inventory_events is the deliberate VASTDB-side overlap with Alice —
# should show her row (actor='alice') alongside Bob's own.
spark.sql(f"SELECT * FROM {DB}.inventory_events ORDER BY event_id").show()

In [ ]:
# Pass/fail auth check: bob-db's VAST Identity Policy grants no access
# to `shipments`, so this write must be REJECTED by VAST directly.
try:
    spark.sql(f"INSERT INTO {DB}.shipments VALUES (999, 1, 'should not be allowed')")
    print('FAIL: Bob was able to write to shipments — isolation is broken')
except Exception as e:
    print('PASS: write to shipments correctly rejected')
    print(type(e).__name__, str(e)[:300])